# Step 3 — Quantization

Quantization reduces model size & inference latency by representing weights
in lower-precision formats (float32 → int8 or int4).

| Method        | Precision | Who maintains | Notes |
|---------------|-----------|---------------|-------|
| BitsAndBytes  | INT8/INT4 | Tim Dettmers  | Zero-effort, runs on GPU; great for quick inference |
| GGUF          | 2–8 bit   | llama.cpp     | CPU-friendly; llama.cpp format; popular for local LLMs |
| AWQ           | INT4      | MIT Han Lab   | Activation-aware; best accuracy at 4-bit |
| ONNX          | FP32/INT8 | Microsoft     | Runtime-agnostic; needed for production serving |

---
**Architecture note**: `roberta2roberta` is encoder-decoder.  
- GGUF / llama.cpp is primarily for *decoder-only* models (GPT, LLaMA).  
  We demonstrate GGUF conversion using the `gguf` Python library + showing
  the conceptual steps; for production you'd use a decoder-only model.  
- BitsAndBytes and ONNX work natively with encoder-decoder.  
- AWQ also primarily targets decoder-only; we show the pattern and note the limitation.

In [ ]:
# !pip install bitsandbytes optimum onnx onnxruntime auto-gptq -q
# For AWQ: !pip install autoawq -q
# For GGUF: !pip install gguf -q  (or build llama.cpp convert scripts)

import os, time, json
import torch
import numpy as np
from transformers import (
    EncoderDecoderModel,
    RobertaTokenizerFast,
    BitsAndBytesConfig,
)
from rouge_score import rouge_scorer

MODEL_DIR = "../models"
MERGED    = f"{MODEL_DIR}/lora_merged"   # output from notebook 01
QUANT_DIR = f"{MODEL_DIR}/quantized"
os.makedirs(QUANT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

tokenizer = RobertaTokenizerFast.from_pretrained(MERGED)

## Helper: Benchmark a model

We measure:
- **Latency** (ms per sample)
- **ROUGE-2** on 50 validation samples
- **Memory** footprint

In [ ]:
import pandas as pd

val_df = pd.read_csv("../data/cnn_dailymail_validation.csv").head(50)

def benchmark_model(model, model_name: str) -> dict:
    """Runs inference on 50 samples, returns latency + ROUGE metrics."""
    scorer   = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    latencies = []
    scores    = {"rouge1": [], "rouge2": [], "rougeL": []}

    model.eval()
    for _, row in val_df.iterrows():
        inputs = tokenizer(
            str(row["article"]),
            return_tensors="pt",
            max_length=512, truncation=True
        ).to(model.device if hasattr(model, 'device') else DEVICE)

        t0 = time.perf_counter()
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=64, num_beams=4)
        latencies.append((time.perf_counter() - t0) * 1000)

        pred = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        ref  = str(row["highlights"])
        s    = scorer.score(ref, pred)
        for k in scores:
            scores[k].append(s[k].fmeasure)

    mem_mb = 0
    for p in model.parameters():
        mem_mb += p.nelement() * p.element_size()
    mem_mb /= 1024 ** 2

    result = {
        "model":      model_name,
        "latency_ms_mean": np.mean(latencies),
        "latency_ms_p95":  np.percentile(latencies, 95),
        "rouge1":    np.mean(scores["rouge1"]),
        "rouge2":    np.mean(scores["rouge2"]),
        "rougeL":    np.mean(scores["rougeL"]),
        "mem_mb":    mem_mb,
    }
    print(json.dumps(result, indent=2))
    return result

all_results = []

## 3a. Baseline — FP32 (no quantization)

In [ ]:
print("Loading FP32 baseline…")
fp32_model = EncoderDecoderModel.from_pretrained(MERGED, torch_dtype=torch.float32)
fp32_model = fp32_model.to(DEVICE)

r = benchmark_model(fp32_model, "FP32 baseline")
all_results.append(r)
del fp32_model; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 3b. BitsAndBytes — INT8 quantization

**How it works**:  
BitsAndBytes (bnb) quantises Linear layer weights to INT8 at load time using
a per-row absmax calibration. The activations remain in FP16/BF16.

```
W_int8[i,j] = round( W_fp32[i,j] / scale_row[i] )
              where scale_row[i] = max(|W_fp32[i,:]|) / 127
```

- Requires `load_in_8bit=True` in `BitsAndBytesConfig`
- Works on CUDA only (uses CUDA kernels for INT8 GEMM)
- Typical: ~2× memory reduction, ~1.1× throughput gain

**INT4 (QLoRA)**: `load_in_4bit=True` with `bnb_4bit_compute_dtype=bfloat16`  
gives ~4× memory reduction using NF4 (Normal Float 4) format.

In [ ]:
if DEVICE == "cuda":
    print("Loading INT8 (BitsAndBytes)…")
    bnb_cfg_8bit = BitsAndBytesConfig(load_in_8bit=True)
    bnb8_model   = EncoderDecoderModel.from_pretrained(
        MERGED,
        quantization_config=bnb_cfg_8bit,
        device_map="auto",
    )
    r = benchmark_model(bnb8_model, "BitsAndBytes INT8")
    all_results.append(r)
    del bnb8_model; torch.cuda.empty_cache()

    print("Loading INT4 NF4 (BitsAndBytes QLoRA-style)…")
    bnb_cfg_4bit = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",          # NormalFloat4 — best for LLM weights
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,     # quantise the quantisation constants too
    )
    bnb4_model = EncoderDecoderModel.from_pretrained(
        MERGED,
        quantization_config=bnb_cfg_4bit,
        device_map="auto",
    )
    r = benchmark_model(bnb4_model, "BitsAndBytes NF4")
    all_results.append(r)
    del bnb4_model; torch.cuda.empty_cache()
else:
    print("⚠  BitsAndBytes requires CUDA — skipping on CPU/MPS.")
    print("   Run this cell on a GPU instance (Colab, Lambda, etc.)")

## 3c. GGUF — llama.cpp format

**What is GGUF?**  
GGUF (GGML Unified Format) is a binary file format used by **llama.cpp** to
store quantised weights. It supports Q4_0, Q4_K_M, Q5_K, Q8_0 etc.

**Workflow**:
```
HuggingFace model
   ↓  (convert_hf_to_gguf.py  from llama.cpp repo)
model.gguf   (float16 — lossless conversion)
   ↓  (llama-quantize)
model-Q4_K_M.gguf  (4-bit quantised)
```

**Primary use case**: CPU inference on consumer hardware via llama.cpp,
Ollama, LM Studio, Jan.  

**Limitation**: llama.cpp supports decoder-only architectures.
For encoder-decoder models (like our roberta2roberta), you would either:
1. Use a decoder-only summarisation model (e.g. `facebook/opt-1.3b` fine-tuned)
2. Export only the decoder half to GGUF

Below we show the conceptual pipeline and export a dummy tensor to GGUF
using the `gguf` Python library.

In [ ]:
# ── GGUF conceptual export ────────────────────────────────────────────────
# Install: pip install gguf

try:
    import gguf
    from gguf import GGUFWriter, GGMLQuantizationType

    gguf_path = f"{QUANT_DIR}/decoder_demo.gguf"
    writer    = GGUFWriter(gguf_path, arch="roberta")

    # Write metadata that llama.cpp expects
    writer.add_name("roberta2roberta-lora-merged")
    writer.add_description("Encoder-decoder demo — decoder weights exported")
    writer.add_uint32("roberta.context_length", 512)
    writer.add_uint32("roberta.embedding_length", 1024)
    writer.add_uint32("roberta.block_count", 24)
    writer.add_uint32("roberta.attention.head_count", 16)

    # Load model and export one weight matrix as example
    model_fp32 = EncoderDecoderModel.from_pretrained(MERGED)
    # Export the first decoder embedding weight
    emb = model_fp32.decoder.roberta.embeddings.word_embeddings.weight.detach().float().numpy()
    writer.add_tensor("token_embd.weight", emb, raw_dtype=GGMLQuantizationType.F32)
    del model_fp32

    writer.write_header_to_file()
    writer.write_kv_data_to_file()
    writer.write_tensors_to_file()
    writer.close()
    print(f"✓ GGUF demo written → {gguf_path}")
    print("  In production: run `llama-quantize model.gguf model-Q4_K_M.gguf Q4_K_M`")

except ImportError:
    print("gguf not installed. Run: pip install gguf")
    print()
    print("Full production GGUF pipeline:")
    print("  1. git clone https://github.com/ggerganov/llama.cpp")
    print("  2. python convert_hf_to_gguf.py ../models/lora_merged --outtype f16")
    print("  3. ./llama-quantize model.gguf model-Q4_K_M.gguf Q4_K_M")
    print("  4. ./llama-cli -m model-Q4_K_M.gguf -p 'Summarize: ...'")

## 3d. AWQ — Activation-Aware Weight Quantization

**Paper**: *AWQ: Activation-aware Weight Quantization for LLM Compression* (MIT, 2023)

**Key insight**: Not all weights are equally important.  
AWQ finds **salient weight channels** (those that correspond to large activations)
and protects them from quantisation error by scaling:

```
Q( W·diag(s) ) · diag(s)⁻¹   ← scale up salient channels before quant,
                                 scale down at runtime → same result, less error
```

**Result**: Better accuracy than GPTQ/BitsAndBytes at INT4, especially for
long-context or knowledge-intensive tasks.

**Limitation**: AutoAWQ targets decoder-only models.  
For encoder-decoder, the pattern is the same but you'd need to apply it
to both encoder and decoder blocks separately.

In [ ]:
# AWQ with AutoAWQ (decoder-only models like LLaMA, Mistral, etc.)
# For demonstration — swap MODEL_PATH to a decoder-only model for full support

try:
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer

    # AWQ works on decoder-only models. To demonstrate the API:
    # (Replace with actual decoder-only model for full run)
    print("AWQ quantization (decoder-only models)")
    print("  Example for a decoder-only model:")
    print("""
    from awq import AutoAWQForCausalLM

    model = AutoAWQForCausalLM.from_pretrained(
        'facebook/opt-1.3b',  # or any decoder-only HF model
        torch_dtype=torch.float16,
        device_map='auto'
    )
    tokenizer = AutoTokenizer.from_pretrained('facebook/opt-1.3b')

    # AWQ quantization config
    quant_config = {
        'zero_point': True,    # use zero-point quantization
        'q_group_size': 128,   # group size for per-group quantization
        'w_bit': 4,            # 4-bit weights
        'version': 'GEMM',     # kernel version (GEMM faster on most GPUs)
    }

    # Calibration + quantization (uses a small calibration dataset)
    model.quantize(tokenizer, quant_config=quant_config)

    # Save
    model.save_quantized('../models/quantized/awq_int4')
    tokenizer.save_pretrained('../models/quantized/awq_int4')

    # Load & infer
    model_awq = AutoAWQForCausalLM.from_quantized('../models/quantized/awq_int4')
    """)
    print("AWQ is installed — update the model_path above to a decoder-only model and run.")

except ImportError:
    print("autoawq not installed. Run: pip install autoawq")
    print("Then use the code pattern shown above.")

## 3e. ONNX Export

**Why ONNX?**  
ONNX (Open Neural Network Exchange) is a runtime-agnostic format.
Once exported you can run the model with:
- `onnxruntime` (CPU/GPU, C++/Python)
- TensorRT (NVIDIA)
- OpenVINO (Intel)
- DirectML (Windows)

**Optimum** (HuggingFace library) handles seq2seq ONNX export properly
by exporting encoder, decoder, and decoder-with-past as separate graphs.

In [ ]:
# ONNX export via HuggingFace Optimum
# pip install optimum[onnxruntime]

try:
    from optimum.onnxruntime import ORTModelForSeq2SeqLM
    from optimum.exporters.onnx import main_export
    from transformers import AutoTokenizer

    onnx_dir = f"{QUANT_DIR}/onnx"
    print("Exporting to ONNX…")

    # Export — creates encoder_model.onnx, decoder_model.onnx, decoder_with_past_model.onnx
    main_export(
        model_name_or_path=MERGED,
        output=onnx_dir,
        task="text2text-generation-with-past",  # seq2seq with KV cache
        optimize="O2",                          # graph-level optimizations
    )
    print(f"✓ ONNX model exported → {onnx_dir}/")

    # Load ONNX model
    ort_model = ORTModelForSeq2SeqLM.from_pretrained(onnx_dir)
    ort_tok    = AutoTokenizer.from_pretrained(MERGED)

    # Benchmark
    r = benchmark_model(ort_model, "ONNX Runtime")
    all_results.append(r)

    # Optional: quantize ONNX model to INT8
    from optimum.onnxruntime import ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig

    quantizer = ORTQuantizer.from_pretrained(onnx_dir)
    q_config  = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
    quantizer.quantize(
        save_dir=f"{QUANT_DIR}/onnx_int8",
        quantization_config=q_config,
    )
    print(f"✓ ONNX INT8 model → {QUANT_DIR}/onnx_int8/")

except ImportError:
    print("optimum[onnxruntime] not installed.")
    print("Run: pip install 'optimum[onnxruntime]'")
    print()
    print("Or manually with torch.onnx.export:")
    print("""
    # Manual ONNX export (encoder only, for illustration)
    import torch.onnx
    model = EncoderDecoderModel.from_pretrained(MERGED).eval()
    dummy = torch.ones(1, 512, dtype=torch.long)
    torch.onnx.export(
        model.encoder,
        (dummy, dummy),                  # (input_ids, attention_mask)
        '../models/quantized/encoder.onnx',
        input_names=['input_ids', 'attention_mask'],
        output_names=['last_hidden_state'],
        dynamic_axes={'input_ids': {0: 'batch', 1: 'seq_len'}},
        opset_version=17,
    )
    """)

## Results Comparison

In [ ]:
if all_results:
    df = pd.DataFrame(all_results)
    print(df[["model", "latency_ms_mean", "latency_ms_p95", "rouge2", "mem_mb"]].to_string(index=False))
    df.to_csv(f"{QUANT_DIR}/quantization_comparison.csv", index=False)
    print(f"\nSaved → {QUANT_DIR}/quantization_comparison.csv")
else:
    print("Run with CUDA GPU to collect all benchmarks.")